## Использование Gamac на реальных данных
Описание реальных задач приведено в [документации](https://github.com/ITMO-CODE-AI/GaMAC/blob/develop/docs/CASE_RU.md)

### Импортируем нужные библиотеки

In [1]:
import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

Примечание: для запуска потребуется около 30-40gb видеопамяти

In [3]:
# Импортируем датафрейм
data = pd.read_csv('../data/synthetic_logs_1kk.csv')

In [5]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=100, target_measures={Internal.BR})

dataset, best_model = gamac.run(table=data)

In [ ]:
# Получение топ-50 меток по лучшему классификатору
print(best_model.model.predict(dataset)[:50])

### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры. 

In [2]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=424x628>,
 <PIL.Image.Image image mode=RGBA size=802x545>,
 <PIL.Image.Image image mode=RGBA size=391x504>,
 <PIL.Image.Image image mode=RGBA size=400x570>,
 <PIL.Image.Image image mode=RGBA size=450x555>]

### Запустим подбор (в ячейке принты скрыты)

In [3]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

dataset, best_model = gamac.run(image=images, text=empty_texts)

/home/user/miniconda3/envs/GaMAC-org/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== Started CVI prediction ===
=== CVI prediction iteration 1/2 ====
=== CVI prediction iteration 2/2 ====
=== Picked MCR in 10.756967067718506s ===
=== MEASURES 0.13705706596374512s, {'MCR': -2.9950725765136115} ===
=== MEASURES 0.09331893920898438s, {'MCR': -2.3441037167739114} ===
=== ALGO 0.36558985710144043s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 0.9220912456512451s, FailedRun, {'bandwidth': 0.06536588142902727, 'max_iter': 101, 'tol': 3.450727222256216e-05} ===
=== ALGO 0.758134126663208s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7562775366070154, 'eps_sq': 0.5719557123763754, 'min_samples': 15} ===
=== ALGO 0.2525618076324463s, FailedRun, {'min_cluster_size': 12, 'min_samples': 10} ===
=== MEASURES 0.12599921226501465s, {'MCR': -198.90139954374936} ===
=== ALGO 0.573908805847168s, SuccessRun, {'threshold': 0.8828107170979665, 'branching_factor': 37, 'n_clusters': 9} ===
=== MEASURES 0.112586975097656

In [4]:
print(best_model.model.predict(dataset))

[2 0 5 2 2 1 0 2 2 2 4 1 2 5 2 3 2 5 5 1 2 1 4 5 2 2 5 5 2 5 5 1 1 2 5 5 2
 5 5 5 5 2 5 2 3 1 5 2 5 5 5 4 5 2 5 5 5 5 2 5 5 3 2 5 2 5 2 5 5 2 4 1 3 5
 5 4 5 2 2 2 1 1 5 5 5 2 1 1 5 1 1 5 5 2 5 5 1 5 5 5 5 2 5 1 5 3 2 2 5 2 2
 4 5 1 1 4 4 2 5 5 5 5 5 5 2 5 1 2 3 2 1 4 2 1 5 2 1 1 2 5 2 5 5 2 5 2 2 2
 4 5 5 2 2 1 5 5 5 5 5 1 2 5 0 4 1 2 2 2 3 3 2 5 5 5 3 5 2 5 5 5 0 5 2 0 5
 1 2 2 2 2 5 5 1 5 5 3 5 5 1 2 5 2 2 3 5 2 5 2 5 5 5 2 2 1 2 5 2 2 5 2 5 1
 5 2 5 2 5 2 2 4 1 5 5 5 5 3 2 4 1 2 2 4 5 5 3 5 2 2 2 5 1 5 1 5 2 2 2 2 5
 4 2 3 1 2 0 2 1 5 1 0 5 5 2 5 1 4 5 5 3 2 2 2 5 1 3 5 1 2 5 5 5 2 5 5 2 2
 2 5 5 2 5 5 5 2 2 3 3 2 2 2 5 5 5 4 1 2 1 5 2 2 2 5 2 2 2 1 1 2 4 5 1 5 5
 3 2 2 0 5 2 5 5 5 3 5 1 5 5 2 1 2 4 0 2 2 4 5 5 5 5 4 5 5 5 5 1 2 5 2 5 5
 2 5 2 2 5 2 2 1 4 5 2 5 2 1 5 3 3 4 5 2 5 2 3 3 4 5 5 1 1 1 5 5 3 5 5 5 2
 5 5 2 3 5 5 2 1 2 5 5 5 2 2 5 1 5 0 5 5 2 5 2 5 5 5 2 2 1 5 2 5 3 2 2 5 4
 2 5 2 1 2 5 5 3 1 1 2 5 5 5 1 5 3 5 0 5 4 5 5 2 2 5 2 1 2 5 5 2 1 3 1 5 2
 4 2 3 2 2 5 5 2 5 3 5 5 